## 1) Imports and Setup

In [1]:
import fitz  # PyMuPDF
import re
import csv
import os
import pandas as pd

# Universal patterns
DATE_PATTERN = r"\b\d{1,2}/\d{1,2}/\d{4}\b"
DAYS_OF_WEEK = ["Δευτέρα", "Τρίτη", "Τετάρτη", "Πέμπτη", "Παρασκευή", "Σάββατο", "Κυριακή"]

# Target files
TARGET_PDFS = [
    "aimolipsies(2014-2024).pdf",
    "anosokatelstalmenoi(2014-2024).pdf",
    "kardiologiki(2014-2024).pdf",
    "nefrologiki(2014-2024).pdf"
]

MASTER_RAW_CSV = "MASTER_RAW_DATA.csv"
MASTER_NAMES_CSV = "MASTER_UNIQUE_NAMES_GENDERS.csv"
MASTER_FINAL_CSV = "MASTER_DATA_FINAL.csv"

## 2) Dynamic Configuration (Sniffer)

In [2]:
CLINIC_CONFIGS = {
    "Default": {
        "id_pattern": r"^\d{5,6}$",
        "markers": ['TSH', 'FT4', 'T3', 'Anti-TPO', 'Anti-TG'],
        "first_page_skip": (5, -2),
        "other_page_skip": (2, -2)
    }
}

## 3) Extractor Functions

In [3]:
def extract_data_from_pdf(pdf_path, config):
    doc = fitz.open(pdf_path)
    extracted_data = []
    numeric_pattern = re.compile(config["id_pattern"]) 

    for page_num in range(len(doc)):
        page = doc[page_num]
        blocks = page.get_text("dict")["blocks"]
        
        all_lines = []
        for block in blocks:
            if "lines" in block:
                for line in block["lines"]:
                    line_text = "".join(span["text"] + " " for span in line["spans"]).strip()
                    all_lines.append(line_text)

        skip_top, skip_bottom = config["first_page_skip"] if page_num == 0 else config["other_page_skip"]
        
        if skip_bottom == 0:
            all_lines = all_lines[skip_top:]
        else:
            all_lines = all_lines[skip_top:skip_bottom]

        current_patient_data = []
        prev_line = ""  # Track the previous line like the old working script
        for line in all_lines:
            if any(line.startswith(day) for day in DAYS_OF_WEEK):
                continue
            if "0.00" in line or "0,00" in line or "€" in line:
                continue

            if numeric_pattern.match(line):
                # ONLY split if the previous line ends with ')' to avoid splitting on patient ID at the bottom
                if prev_line.endswith(')'):
                    if current_patient_data:
                        extracted_data.append(current_patient_data)
                        current_patient_data = []
            
            current_patient_data.append(line)
            prev_line = line

        if current_patient_data:
            extracted_data.append(current_patient_data)

    return extracted_data

def extract_patient_name(patient_data, config):
    # Make a copy so we don't modify the passed list weirdly during iterations
    data = patient_data.copy()
    
    exam_id = data.pop(0) if data else None

    name_parts = []
    for i, item in enumerate(data):
        if re.search(DATE_PATTERN, item):
            name_parts = data[:i]
            break

    if name_parts:
        for _ in range(len(name_parts)):
            if data:
                data.pop(0)

    name_part = " ".join(name_parts).strip() if name_parts else ""
    full_name, father_name, patient_am = "", "", ""

    if name_part:
        if "(" in name_part:
            full_name = name_part.split("(")[0].strip()
            father_name_match = re.search(r"\((.*?)\)", name_part)
            if father_name_match:
                father_name = father_name_match.group(1).strip().replace(" ", "")
            patient_am_match = re.search(r"-(\d[\d ]*\d)", name_part)
            if patient_am_match:
                patient_am = patient_am_match.group(1).replace(" ", "")
        else:
            full_name = name_part.strip()

    full_name = full_name.split(" ")
    last_name = full_name.pop(0) if full_name else ""
    first_name = full_name[::-1].pop(0) if full_name else ""

    date_of_exam = data.pop(0) if len(data) > 0 else None
    date_of_birth = data.pop(0) if len(data) > 0 else None

    patient_id_index = -1
    for i, item in enumerate(data):
        if re.match(r"^\d+$", item):  # Search for the first purely numeric string
            patient_id_index = i
            break
            
    if patient_id_index != -1:
        # Everything before the ID is the clinic name
        clinic_name = " ".join(data[:patient_id_index]).strip()
        # The numeric string is the Patient ID
        patient_id = data[patient_id_index]
        
        # Remove the extracted Clinic Name and Patient ID from the data list
        for _ in range(patient_id_index + 1):
            if data:
                data.pop(0)
    else:
        # Fallback just in case no number is found
        clinic_name = " ".join(data[:2]).strip() if len(data) >= 2 else ""
        if len(data) >= 2:
            [data.pop(0) for _ in range(2)]
        patient_id = data.pop(0) if len(data) > 0 else None


    data.reverse()
    data = [re.sub(r"\(.*?\)", "", item).strip() for item in data]

    # Initialize markers dictionary based on config
    marker_results = {marker: None for marker in config["markers"]}

    for i in range(0, len(data), 2):
        marker = data[i]
        if i + 1 < len(data):
            value = data[i + 1].replace(',', '.')
            if any(char in value for char in ['<', '>']):
                float_value = value.strip()
            else:
                try:
                    float_value = float(value) if value else None
                except ValueError:
                    float_value = None

            if marker in marker_results:
                marker_results[marker] = float_value

    return {
        "ID": patient_id, "First Name": first_name, "Last Name": last_name, 
        "Father Name": father_name, "AM": patient_am, "Date": date_of_exam, 
        "Date of Birth": date_of_birth, "Clinic Name": clinic_name, 
        "Exam ID": exam_id, **marker_results
    }

## 4) Batch Extractor

In [4]:
all_processed_data = []
all_unique_names = set()

active_config = CLINIC_CONFIGS["Default"]

for pdf_path in TARGET_PDFS:
    if not os.path.exists(pdf_path):
        print(f"Skipping {pdf_path} (File not found in directory)")
        continue
        
    print(f"Processing: {pdf_path}...")

    # Extract Data directly
    raw_patient_data = extract_data_from_pdf(pdf_path, active_config)

    for patient in raw_patient_data:
        details = extract_patient_name(patient, active_config)
        
        # Clean/Skip bad entries
        if details["First Name"] == "" or details["AM"] == "":
            continue
        if details["Date of Birth"] and not re.match(DATE_PATTERN, details["Date of Birth"]):
            continue
            
        all_processed_data.append(details)
        all_unique_names.add(details["First Name"])

# Save Master Raw Data
df_raw = pd.DataFrame(all_processed_data)
df_raw.to_csv(MASTER_RAW_CSV, index=False, quoting=csv.QUOTE_ALL, encoding='utf-8')
print(f"\nAll data extracted and saved to {MASTER_RAW_CSV}")
print(f"Total valid records: {len(df_raw)}")

# Save Master Unique Names
names_df = pd.DataFrame({"Name": sorted(list(all_unique_names)), "Gender": ""})

if os.path.exists(MASTER_NAMES_CSV):
    existing_genders = pd.read_csv(MASTER_NAMES_CSV)
    if 'Name' in existing_genders.columns and 'Gender' in existing_genders.columns:
        names_df = pd.merge(names_df, existing_genders, on='Name', how='left', suffixes=('_new', ''))
        names_df['Gender'] = names_df['Gender'].combine_first(names_df['Gender_new'])
        names_df = names_df[['Name', 'Gender']]

names_df.to_csv(MASTER_NAMES_CSV, index=False, encoding='utf-8')
print(f"Unique names saved to {MASTER_NAMES_CSV}")

Processing: aimolipsies(2014-2024).pdf...
Processing: anosokatelstalmenoi(2014-2024).pdf...
Processing: kardiologiki(2014-2024).pdf...
Processing: nefrologiki(2014-2024).pdf...

All data extracted and saved to MASTER_RAW_DATA.csv
Total valid records: 48280
Unique names saved to MASTER_UNIQUE_NAMES_GENDERS.csv


# 5) Master Merger 


In [5]:
try:
    # 1. Read the master raw data and the filled-out gender data
    master_df = pd.read_csv(MASTER_RAW_CSV)
    gender_df = pd.read_csv(MASTER_NAMES_CSV)
    
    # 2. Create a dictionary mapping Name -> Gender
    gender_dict = dict(zip(gender_df['Name'], gender_df['Gender']))
    
    # 3. Apply the master gender mapping
    master_df['Gender'] = master_df['First Name'].map(gender_dict).fillna("Unknown")
    
    # 4. Save the final product
    master_df.to_csv(MASTER_FINAL_CSV, index=False, quoting=csv.QUOTE_ALL, encoding='utf-8')
    
    print("Success. Final master data with genders saved to:")
    print(MASTER_FINAL_CSV)
    print(f"Total Rows: {len(master_df)}")
    
    
except FileNotFoundError as e:
    print(f"Error: Could not find required file. {e}")

Success. Final master data with genders saved to:
MASTER_DATA_FINAL.csv
Total Rows: 48280
